[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ThanhDatVN/AI_journey/blob/main/notebooks/26_recommender_systems/03_matrix_factorization.ipynb)

# Bài 03 — Matrix factorization

**🎯 Mục tiêu**
- Mô hình **nhân tố ẩn**: $\hat r_{ui} = \mu + b_u + b_i + \mathbf p_u^\top\mathbf q_i$; liên hệ với SVD (chương 21) và vì sao không dùng thẳng SVD khi thiếu dữ liệu.
- Huấn luyện bằng **SGD** (Funk SVD) và **ALS** from scratch; regularization, early stopping, số nhân tố $k$.
- Diễn giải nhân tố ẩn; **PureSVD** cho top-N trên phản hồi ngầm.
- Template model cho matrix factorization.

In [ ]:
# ⚙️ Setup — chạy được cả local lẫn Google Colab (local: đã `pip install -e .` thì bỏ qua)
import sys
if "google.colab" in sys.modules:
    !test -d /content/ml-course || git clone -q --depth 1 https://github.com/ThanhDatVN/AI_journey /content/ml-course
    !pip install -q -e /content/ml-course
    sys.path.insert(0, "/content/ml-course/src")
    %cd /content/ml-course/notebooks/26_recommender_systems

In [ ]:
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score

from ml_course.metrics import ndcg_at_k, precision_at_k, recall_at_k
from ml_course.visualization import set_style

set_style()

In [ ]:
ratings = pd.read_csv("../../data/sample/ratings.csv")
items = pd.read_csv("../../data/sample/items.csv")
n_users, n_items = ratings.user_id.max() + 1, ratings.item_id.max() + 1


def split_per_user(ratings, test_frac=0.2, min_ratings=5, seed=0):
    rng = np.random.default_rng(seed)
    test_mask = np.zeros(len(ratings), bool)
    for _, g in ratings.groupby("user_id"):
        if len(g) >= min_ratings:
            test_mask[rng.choice(g.index, max(1, int(test_frac * len(g))), replace=False)] = True
    return ratings[~test_mask], ratings[test_mask]


train, test = split_per_user(ratings)
u_tr, i_tr, y_tr = train.user_id.to_numpy(), train.item_id.to_numpy(), train.rating.to_numpy(float)
u_te, i_te, y_te = test.user_id.to_numpy(), test.item_id.to_numpy(), test.rating.to_numpy(float)
mu = y_tr.mean()


def rmse(pred, y=y_te):
    return float(np.sqrt(np.mean((y - np.clip(pred, 1, 5)) ** 2)))

## 1. Ý tưởng: user và item trong cùng một không gian ẩn

Mỗi user là vector $\mathbf p_u \in \mathbb R^k$ ("mức thích hành động, lãng mạn, phim cũ..."), mỗi item là $\mathbf q_i \in \mathbb R^k$ ("mức hành động, lãng mạn, cũ..."). Độ hợp = tích vô hướng:
$$\hat r_{ui} = \mu + b_u + b_i + \mathbf p_u^\top\mathbf q_i$$
Ma trận rating $\approx$ tích hai ma trận "gầy" $P Q^\top$ (hạng $k$) — như **SVD cắt cụt** (chương 21). Nhưng SVD cần ma trận **đầy đủ**; 91% ô đang trống. Điền 0 coi "chưa xem" như "ghét" → sai. Thay vào đó, chỉ tối ưu trên các ô **quan sát được**:
$$\min_{P, Q, b}\sum_{(u,i) \in \mathcal K}\big(r_{ui} - \hat r_{ui}\big)^2 + \lambda\big(\lVert\mathbf p_u\rVert^2 + \lVert\mathbf q_i\rVert^2 + b_u^2 + b_i^2\big)$$

## 2. SGD (Funk SVD)

Với mỗi rating quan sát, sai số $e_{ui} = r_{ui} - \hat r_{ui}$, cập nhật:
$$b_u \mathrel{+}= \eta(e_{ui} - \lambda b_u), \quad b_i \mathrel{+}= \eta(e_{ui} - \lambda b_i), \quad \mathbf p_u \mathrel{+}= \eta(e_{ui}\mathbf q_i - \lambda\mathbf p_u), \quad \mathbf q_i \mathrel{+}= \eta(e_{ui}\mathbf p_u - \lambda\mathbf q_i)$$

In [ ]:
def mf_sgd(k=20, lr=0.01, reg=0.1, epochs=60, seed=0, track=True):
    r = np.random.default_rng(seed)
    P, Q = r.normal(0, 0.1, (n_users, k)), r.normal(0, 0.1, (n_items, k))
    bu, bi = np.zeros(n_users), np.zeros(n_items)
    history = []
    for _ in range(epochs):
        for j in r.permutation(len(y_tr)):
            u, i = u_tr[j], i_tr[j]
            e = y_tr[j] - (mu + bu[u] + bi[i] + P[u] @ Q[i])
            bu[u] += lr * (e - reg * bu[u])
            bi[i] += lr * (e - reg * bi[i])
            pu = P[u].copy()
            P[u] += lr * (e * Q[i] - reg * P[u])
            Q[i] += lr * (e * pu - reg * Q[i])
        if track:
            pred = lambda uu, ii: mu + bu[uu] + bi[ii] + np.sum(P[uu] * Q[ii], 1)
            history.append((rmse(pred(u_tr, i_tr), y_tr), rmse(pred(u_te, i_te))))
    return P, Q, bu, bi, np.array(history)


t0 = time.perf_counter()
runs = {reg: mf_sgd(k=20, reg=reg) for reg in [0.02, 0.1]}
print(f"thời gian: {(time.perf_counter() - t0) / 2:.1f}s mỗi lần train (60 epoch, vòng lặp Python)")
fig, ax = plt.subplots(figsize=(10, 4))
for reg, (*_, hist) in runs.items():
    ax.plot(hist[:, 0], ls="--", label=f"train, λ = {reg}")
    ax.plot(hist[:, 1], label=f"test, λ = {reg}")
ax.set_xlabel("epoch")
ax.set_ylabel("RMSE")
ax.set_ylim(0.3, 0.75)
ax.legend()
plt.show()
for reg, (*_, hist) in runs.items():
    best = int(np.argmin(hist[:, 1]))
    print(f"λ = {reg:<5}: RMSE test tốt nhất {hist[best, 1]:.4f} ở epoch {best + 1} | epoch 60: train {hist[-1, 0]:.4f}, test {hist[-1, 1]:.4f}")

Với $\lambda$ nhỏ, RMSE train tiếp tục giảm nhưng RMSE test tăng trở lại sau vài chục epoch — **overfitting**: mỗi user chỉ có ~15 rating mà phải ước lượng 20 nhân tố. Regularization lớn hơn (hoặc **early stopping** trên validation) giữ mô hình ổn định.

## 3. ALS (Alternating Least Squares)

Cố định $Q$ → mỗi $\mathbf p_u$ là một bài toán **Ridge** độc lập (nghiệm đóng); rồi cố định $P$ → giải từng $\mathbf q_i$. Lặp lại:
$$\mathbf p_u = (Q_{I_u}^\top Q_{I_u} + \lambda I)^{-1}Q_{I_u}^\top\mathbf r_u$$
Song song hoá dễ (mỗi user độc lập) — lý do Spark MLlib dùng ALS. Ở đây áp dụng trên phần dư sau baseline $\mu + b_u + b_i$ (bài 02).

In [ ]:
def fit_biases(lam=5.0, n_iter=15):
    bu, bi = np.zeros(n_users), np.zeros(n_items)
    for _ in range(n_iter):
        bi = np.bincount(i_tr, y_tr - mu - bu[u_tr], n_items) / (lam + np.bincount(i_tr, minlength=n_items))
        bu = np.bincount(u_tr, y_tr - mu - bi[i_tr], n_users) / (lam + np.bincount(u_tr, minlength=n_users))
    return bu, bi


def mf_als(k=10, lam=5.0, n_iter=15, seed=0):
    bu, bi = fit_biases()
    resid = y_tr - (mu + bu[u_tr] + bi[i_tr])
    r = np.random.default_rng(seed)
    P, Q = r.normal(0, 0.1, (n_users, k)), r.normal(0, 0.1, (n_items, k))
    by_user = [np.flatnonzero(u_tr == u) for u in range(n_users)]
    by_item = [np.flatnonzero(i_tr == i) for i in range(n_items)]
    history = []
    for _ in range(n_iter):
        for u, idx in enumerate(by_user):
            Qi = Q[i_tr[idx]]
            P[u] = np.linalg.solve(Qi.T @ Qi + lam * np.eye(k), Qi.T @ resid[idx])
        for i, idx in enumerate(by_item):
            Pu = P[u_tr[idx]]
            Q[i] = np.linalg.solve(Pu.T @ Pu + lam * np.eye(k), Pu.T @ resid[idx])
        history.append(rmse(mu + bu[u_te] + bi[i_te] + np.sum(P[u_te] * Q[i_te], 1)))
    return P, Q, bu, bi, history


for lam in [1.0, 5.0, 20.0]:
    t0 = time.perf_counter()
    *_, hist_als = mf_als(lam=lam)
    print(f"ALS λ = {lam:<4}: RMSE test sau 15 vòng = {hist_als[-1]:.4f} | thời gian {time.perf_counter() - t0:.2f}s")

ALS hội tụ sau vài vòng và nhanh hơn nhiều so với SGD bằng vòng lặp Python (mỗi bước là phép toán ma trận nhỏ). Cả hai cho RMSE ~0.61–0.62 — tốt hơn baseline 0.631 một chút, tương đương KNN (bài 02).

## 4. Số nhân tố $k$

In [ ]:
rows = []
for k in [1, 2, 5, 10, 20, 40]:
    *_, hist_k = mf_als(k=k, lam=5.0, n_iter=10)
    rows.append({"k": k, "RMSE test (ALS, λ = 5)": hist_k[-1]})
pd.DataFrame(rows).set_index("k").round(4)

Với regularization phù hợp, RMSE ít nhạy với $k$ — phần lớn cải thiện đã nằm ở thiên lệch, và mỗi user có quá ít rating để ước lượng nhiều nhân tố. Chọn $k$, $\lambda$ bằng validation (chương 16).

## 5. Nhân tố ẩn có nghĩa gì?

Không ai nói cho mô hình biết thể loại phim. Chiếu vector item $\mathbf q_i$ xuống 2 chiều (PCA) và tô màu theo thể loại:

In [ ]:
P_als, Q_als, *_ = mf_als(k=10, lam=5.0)
active = np.bincount(i_tr, minlength=n_items) >= 15                     # phim đủ rating để vector đáng tin
Z = PCA(2).fit_transform(Q_als[active])
genres = items.genre.to_numpy()[active]
fig, ax = plt.subplots(figsize=(8, 6))
for g in sorted(set(genres)):
    ax.scatter(*Z[genres == g].T, s=25, label=g)
ax.legend(fontsize=8)
ax.set_title("Vector nhân tố của phim (PCA 2D) — tô màu theo thể loại")
plt.show()
acc_q = cross_val_score(LogisticRegression(max_iter=3000), Q_als[active], genres, cv=5).mean()
majority = pd.Series(genres).value_counts(normalize=True).max()
print(f"đoán thể loại từ vector nhân tố 10 chiều: accuracy CV = {acc_q:.2f} (đoán lớp nhiều nhất: {majority:.2f})")

Các thể loại có xu hướng tụ thành vùng (chồng lấn nhiều): từ vector nhân tố đoán được thể loại tốt hơn hẳn đoán mò — nhân tố ẩn **tự phát hiện** một phần cấu trúc nội dung chỉ từ hành vi. Tuy vậy, nhân tố không có tên — chúng là tổ hợp của nhiều khía cạnh (thể loại chỉ là khía cạnh nổi bật nhất); diễn giải chỉ mang tính gợi ý.

## 6. Top-N: PureSVD trên phản hồi ngầm

Cho bài toán **chọn xem gì**, ta phân rã ma trận nhị phân "đã xem" $B$ (điền 0 cho ô trống — ở đây đúng nghĩa "chưa xem") bằng SVD cắt cụt: $B \approx U_k\Sigma_kV_k^\top$, điểm $s_{ui} = (U_k\Sigma_kV_k^\top)_{ui}$ (Cremonesi et al., 2010).

In [ ]:
B = np.zeros((n_users, n_items))
B[u_tr, i_tr] = 1.0
seen = {u: set(g.item_id) for u, g in train.groupby("user_id")}
test_ratings = {u: dict(zip(g.item_id, g.rating)) for u, g in test.groupby("user_id")}


def evaluate_topk(score_fn, k=10):
    prec, rec, ndcg = [], [], []
    for u, rated in test_ratings.items():
        liked = [i for i, r in rated.items() if r >= 4]
        if not liked:
            continue
        scores = np.asarray(score_fn(u), float).copy()
        scores[list(seen.get(u, []))] = -np.inf
        top = np.argsort(-scores)[:k]
        prec.append(precision_at_k(top, liked, k))
        rec.append(recall_at_k(top, liked, k))
        ndcg.append(ndcg_at_k(top, {i: r - 3 for i, r in rated.items() if r >= 4}, k))
    return {"precision@10": np.mean(prec), "recall@10": np.mean(rec), "NDCG@10": np.mean(ndcg)}


U_b, s_b, Vt_b = np.linalg.svd(B, full_matrices=False)
topn = {"popularity": evaluate_topk(lambda u: B.sum(0))}
for k in [3, 5, 10, 20]:
    B_k = (U_b[:, :k] * s_b[:k]) @ Vt_b[:k]
    topn[f"PureSVD k = {k}"] = evaluate_topk(lambda u, B_k=B_k: B_k[u])
pd.DataFrame(topn).T.round(4)

PureSVD với $k$ nhỏ là mô hình top-N tốt nhất đến giờ (so với item/user-KNN ở bài 02). $k$ lớn → tái tạo gần đúng ma trận đã xem → gợi ý gần như ngẫu nhiên các phim lẻ; $k$ nhỏ giữ lại các "trục" chính của hành vi (phổ biến + gu).

Với dữ liệu ngầm lớn, các phương pháp chuyên dụng: **ALS cho phản hồi ngầm** (Hu, Koren & Volinsky, 2008 — trọng số tin cậy cho mọi ô, kể cả ô 0), **BPR** (Rendle et al., 2009 — tối ưu trực tiếp thứ hạng: item đã xem phải xếp trên item chưa xem), và các mô hình nơ-ron (two-tower, chương 39).

## 7. Template model: Matrix Factorization

| Mục | Nội dung |
|---|---|
| Problem | dự đoán rating (tường minh) hoặc xếp hạng (ngầm) |
| Intuition | user và item cùng sống trong không gian "gu" ít chiều |
| Math | $\hat r_{ui} = \mu + b_u + b_i + \mathbf p_u^\top\mathbf q_i$, tối thiểu bình phương sai số + L2 trên ô quan sát |
| Algorithm | SGD (Funk), ALS (song song), SVD cắt cụt (PureSVD cho ngầm) |
| Complexity | SGD: $O(|\mathcal K|k)$ mỗi epoch; ALS: $O(|\mathcal K|k^2 + (n_u + n_i)k^3)$ mỗi vòng |
| Hyperparameters | $k$, $\lambda$, learning rate, số epoch (early stopping) |
| Assumptions | ma trận gần hạng thấp; dữ liệu thiếu ngẫu nhiên (thường **không** đúng — bài 04) |
| Strengths | gọn, mạnh, mở rộng tốt, thêm được bias/thời gian/feature |
| Weaknesses | cold start; nhân tố khó diễn giải; tối ưu RMSE ≠ tối ưu xếp hạng |
| Failure cases | overfit khi ít rating/user; $k$ lớn không regularization |
| Production | ALS phân tán (Spark), ANN cho tìm kiếm top-N, cập nhật định kỳ |
| Alternatives | KNN, BPR, implicit ALS, factorization machines, two-tower neural |

## 🧠 Kiểm tra nhanh

1. Vì sao không áp dụng thẳng SVD lên ma trận rating có ô trống?
2. Viết quy tắc cập nhật SGD cho $\mathbf p_u$, $\mathbf q_i$.
3. Vì sao ALS dễ song song hoá? Mỗi bước giải bài toán gì?
4. Dấu hiệu overfitting trong MF? Cách chống?
5. PureSVD khác MF cho rating ở đâu? Vì sao $k$ lớn lại kém?

## 📌 Tóm tắt

- MF: user/item thành vector ẩn; chỉ tối ưu trên ô quan sát, có regularization và thiên lệch.
- SGD linh hoạt; ALS nhanh, song song; RMSE ít nhạy với $k$ khi đã regularize.
- Nhân tố ẩn bắt được cấu trúc nội dung; cho top-N dùng phân rã phản hồi ngầm (PureSVD, implicit ALS, BPR).

## 📚 Tài liệu tham khảo

- Koren, Bell & Volinsky (2009) — *Matrix Factorization Techniques for Recommender Systems*
- Hu, Koren & Volinsky (2008) — *Collaborative Filtering for Implicit Feedback Datasets*
- Cremonesi, Koren & Turrin (2010) — *Performance of Recommender Algorithms on Top-N Recommendation Tasks*
- Rendle et al. (2009) — *BPR: Bayesian Personalized Ranking from Implicit Feedback*